# TinyImageNet Canonical ALiBi Re-training (1D + 2D, seed 42)

Trains **both 1D-ALiBi and 2D-ALiBi** on TinyImageNet (seed 42) under
the same canonical protocol used for the CIFAR-100 12-seed re-training, so
the per-seed (1D, 2D) pair is bit-comparable: the only experimental
variable is the distance metric in the ALiBi bias.

**Output**: `/content/drive/MyDrive/Trained models_TinyImageNet_v2/`
(old `Trained models_TinyImageNet/` is left untouched as backup)

**Compute**: ~22 h / seed × 2 PE types = **~44 h** for the default
SEEDS=[42] setting. Easy to extend to [42, 123, 456] by editing the
SEEDS list in `tinyimagenet_alibi_canonical.py`. Has resume logic, so
you can stop/restart across multiple Colab sessions.

**Prerequisites on Drive**:
- `apply_2d_alibi_patch.py`
- `full_scale_experiment.py` (unpatched original)
- `tinyimagenet_alibi_canonical.py` (this run's training script)


## 1. Mount Drive + GPU check

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
assert torch.cuda.is_available(), 'No GPU!'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'PyTorch: {torch.__version__}')


## 2. Install deps + stage scripts + (re)generate patched module

Copies the three required files from Drive into `/content/` and re-runs
the patcher to regenerate `full_scale_experiment_v2.py`. The patched file
is a generated artifact and is NOT kept on Drive -- always regenerated
here so we never accidentally use a stale copy.

In [ ]:
!pip install -q timm tqdm

import os, shutil

DRIVE_BASE = '/content/drive/MyDrive'
PE_EXPERIMENT_BASE = os.path.join(DRIVE_BASE, 'pe_experiment')

# Each entry: (filename_in_content, list_of_candidate_paths_on_drive)
needed = [
    ('apply_2d_alibi_patch.py', [
        os.path.join(DRIVE_BASE, 'apply_2d_alibi_patch.py'),
        os.path.join(PE_EXPERIMENT_BASE, 'apply_2d_alibi_patch.py'),
    ]),
    ('full_scale_experiment.py', [
        os.path.join(PE_EXPERIMENT_BASE, 'full_scale_experiment.py'),
        os.path.join(DRIVE_BASE, 'full_scale_experiment.py'),
    ]),
    ('tinyimagenet_alibi_canonical.py', [
        os.path.join(DRIVE_BASE, 'tinyimagenet_alibi_canonical.py'),
        os.path.join(PE_EXPERIMENT_BASE, 'tinyimagenet_alibi_canonical.py'),
    ]),
]

for fname, candidates in needed:
    src = next((p for p in candidates if os.path.isfile(p)), None)
    if src is None:
        raise FileNotFoundError(
            f'Cannot find {fname} in any of: {candidates}')
    shutil.copy(src, f'/content/{fname}')
    print(f'  Copied: {src} -> /content/{fname}')

# Regenerate the patched module (do not trust any cached copy)
exit_code = os.system(
    'cd /content && python apply_2d_alibi_patch.py '
    'full_scale_experiment.py full_scale_experiment_v2.py')
if exit_code != 0:
    raise RuntimeError(f'Patch failed with exit code {exit_code}')

assert os.path.isfile('/content/full_scale_experiment_v2.py')
print('\nReady.')


## 3. Run training (1D-ALiBi then 2D-ALiBi, seed 42)

The script iterates **seed-major**: for each seed it trains 1D-ALiBi first,
then 2D-ALiBi. With the default SEEDS=[42] this is just two runs,
back-to-back, so the pair shares the same CUDA driver state -- strongest
possible form of pairing.

Any run already completed (300 epochs done) is auto-skipped, so you can
interrupt this cell and re-launch it after a Colab session resets.

**Note**: First run will trigger TinyImageNet download (~250 MB into Drive)
and val/ re-organisation. Subsequent runs reuse the prepared dataset.

In [ ]:
import subprocess, sys

cmd = 'cd /content && python -u tinyimagenet_alibi_canonical.py'

print('=' * 70)
print('TinyImageNet canonical ALiBi training (1D + 2D, seed 42)')
print('=' * 70)

p = subprocess.Popen(
    cmd, shell=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    bufsize=1, universal_newlines=True,
)
for line in p.stdout:
    print(line, end='', flush=True)
    sys.stdout.flush()
exit_code = p.wait()

print('\n' + '=' * 70)
print('ALL DONE' if exit_code == 0 else f'Exited with code {exit_code}')
print('=' * 70)


## 4. Status check (run any time)

Shows per-seed best accuracy and the paired 2D-vs-1D delta. Safe to re-run
during partial training; only fully-completed seeds (300 epochs) appear in
the paired-difference summary.

Also compares to the **legacy** TIN result (under the original
`Trained models_TinyImageNet/` directory) so you can sanity-check how
different the canonical 1D-ALiBi number is from the old single-seed entry
(reported as 53.41% in the manuscript's Table 3).

In [ ]:
import os, json

V2_RESULTS  = '/content/drive/MyDrive/Trained models_TinyImageNet_v2'
OLD_RESULTS = '/content/drive/MyDrive/Trained models_TinyImageNet'
SEEDS = [42]   # extend here if you trained more seeds

def best_acc(results_dir, pe, seed, min_epochs=300):
    p = os.path.join(results_dir, f'{pe}_seed{seed}', 'training_history.json')
    if not os.path.isfile(p):
        return None, 0
    with open(p) as f:
        h = json.load(f)
    accs = h.get('val_acc', [])
    return (max(accs) if accs else None), len(accs)

print('TinyImageNet canonical ALiBi training status:')
print('=' * 70)
for seed in SEEDS:
    print(f'\nSeed {seed}:')
    a1, n1 = best_acc(V2_RESULTS, 'alibi',    seed)
    a2, n2 = best_acc(V2_RESULTS, 'alibi_2d', seed)
    s1 = f'{a1:.2f}%' if a1 is not None else '   --'
    s2 = f'{a2:.2f}%' if a2 is not None else '   --'
    st1 = 'DONE' if n1 >= 300 else (f'partial {n1}/300' if n1 > 0 else 'not started')
    st2 = 'DONE' if n2 >= 300 else (f'partial {n2}/300' if n2 > 0 else 'not started')
    print(f'  1D-ALiBi (canonical): {s1:>8}  [{st1}]')
    print(f'  2D-ALiBi (canonical): {s2:>8}  [{st2}]')
    if a1 is not None and a2 is not None and n1 >= 300 and n2 >= 300:
        d = a2 - a1
        print(f'  --> Paired delta (2D - 1D): {d:+.2f} pp')

# Compare canonical 1D-ALiBi to legacy 1D-ALiBi on the same seed
print('\n' + '-' * 70)
print('Sanity check vs. legacy (original protocol) 1D-ALiBi TIN result:')
for seed in SEEDS:
    a_canon, n_canon = best_acc(V2_RESULTS,  'alibi', seed)
    a_legacy, n_legacy = best_acc(OLD_RESULTS, 'alibi', seed)
    if a_canon is not None and a_legacy is not None:
        delta = a_canon - a_legacy
        print(f'  seed {seed}: canonical={a_canon:.2f}%  '
              f'legacy={a_legacy:.2f}%  diff={delta:+.2f} pp')
        if abs(delta) > 2.0:
            print(f'     [!] Difference > 2 pp -- worth investigating which '
                  f'training detail caused it before proceeding.')
    elif a_legacy is not None:
        print(f'  seed {seed}: legacy={a_legacy:.2f}%, canonical not yet done')
    else:
        print(f'  seed {seed}: no comparison possible yet')
